In [8]:
from pathlib import Path
import shutil

# 1) Find the real project root (the folder that already has data/, src/, notebooks/)
def find_project_root():
    here = Path.cwd().resolve()
    for p in [here] + list(here.parents):
        if (p / "data").exists() and (p / "src").exists() and (p / "notebooks").exists():
            return p
    # fallback: if you're inside project/notebooks, project root is parent
    return Path.cwd().resolve().parent

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR  = DATA_DIR / "raw"
PROC_DIR = DATA_DIR / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)

print("USING PATHS:")
print("  PROJECT_ROOT:", PROJECT_ROOT)
print("  RAW_DIR     :", RAW_DIR)
print("  PROC_DIR    :", PROC_DIR)

# 2) If a wrong tree exists like: <cwd>/project/data/{raw,processed}, move its files into the correct place.
wrong_root = Path.cwd() / "project"
wrong_raw  = wrong_root / "data" / "raw"
wrong_pro  = wrong_root / "data" / "processed"

moved = []
for src_dir, dst_dir in [(wrong_raw, RAW_DIR), (wrong_pro, PROC_DIR)]:
    if src_dir.exists():
        for p in src_dir.glob("*"):
            try:
                shutil.move(str(p), str(dst_dir / p.name))
                moved.append(str(p))
            except Exception as e:
                print("Skip moving", p, "->", e)

# Optionally clean up empty folders
for d in [wrong_raw, wrong_pro, wrong_root]:
    try:
        if d.exists() and d.is_dir() and not any(d.iterdir()):
            d.rmdir()
    except Exception:
        pass

if moved:
    print("Moved files from wrong location:", moved)
else:
    print("No stray files found under notebooks/project/.")


USING PATHS:
  PROJECT_ROOT: C:\Users\User\bootcamp_Khushi_Khanna\project
  RAW_DIR     : C:\Users\User\bootcamp_Khushi_Khanna\project\data\raw
  PROC_DIR    : C:\Users\User\bootcamp_Khushi_Khanna\project\data\processed
Moved files from wrong location: ['C:\\Users\\User\\bootcamp_Khushi_Khanna\\project\\notebooks\\project\\data\\raw\\api_revenue_banks_20250825-1455.csv', 'C:\\Users\\User\\bootcamp_Khushi_Khanna\\project\\notebooks\\project\\data\\processed\\banks_quarterly_revenue_20250825-1455.csv', 'C:\\Users\\User\\bootcamp_Khushi_Khanna\\project\\notebooks\\project\\data\\processed\\banks_quarterly_revenue_20250825-1455.parquet']


In [15]:
import sys, subprocess, importlib, datetime as dt
from io import StringIO
tables = pd.read_html(StringIO(html), flavor="lxml")  # instead of pd.read_html(html, ...)

def ensure(pkg):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ("pandas", "numpy", "requests", "yfinance", "lxml", "pyarrow"):
    ensure(pkg)

import pandas as pd, numpy as np, requests, yfinance as yf
STAMP = dt.datetime.now().strftime("%Y%m%d-%H%M")

BANK_TICKERS = {
    "JPMorgan": "JPM",
    "Bank of America": "BAC",
    "Citigroup": "C",
    "Goldman Sachs": "GS",
    "Morgan Stanley": "MS",
    "Deutsche Bank": "DB",
}

print("Ready. RAW_DIR:", RAW_DIR, "PROC_DIR:", PROC_DIR)


Ready. RAW_DIR: C:\Users\User\bootcamp_Khushi_Khanna\project\data\raw PROC_DIR: C:\Users\User\bootcamp_Khushi_Khanna\project\data\processed


In [16]:
def _rename_revenue(df: pd.DataFrame) -> pd.DataFrame:
    if "revenue" in df.columns:
        return df
    for c in ["Total Revenue","totalRevenue","Revenue","Total revenue","TotalRevenue","sales","Sales"]:
        if c in df.columns:
            return df.rename(columns={c: "revenue"})
    for c in df.columns:
        if "revenue" in str(c).lower():
            return df.rename(columns={c: "revenue"})
    return df

def fetch_quarterly_revenue_yf(ticker: str, limit: int = 40) -> pd.DataFrame:
    t = yf.Ticker(ticker)
    qf = t.quarterly_financials
    if qf is None or qf.empty:
        raise RuntimeError(f"No quarterly financials for {ticker}.")
    df = qf.T.reset_index().rename(columns={"index": "date"})
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = _rename_revenue(df)
    if "revenue" not in df.columns:
        raise RuntimeError(f"Revenue column not found for {ticker}.")
    out = (df[["date","revenue"]]
           .dropna(subset=["date"])
           .sort_values("date")
           .tail(limit)
           .reset_index(drop=True))
    out["source"] = "yfinance"
    return out

frames = []
for bank, tkr in BANK_TICKERS.items():
    try:
        dfi = fetch_quarterly_revenue_yf(tkr, limit=40)
        dfi["bank"] = bank
        dfi["ticker"] = tkr
        frames.append(dfi)
        print(f"OK {tkr}: {len(dfi)} rows")
    except Exception as e:
        print(f"FAIL {tkr}: {e}")

if not frames:
    raise RuntimeError("No data fetched.")

rev_all = (pd.concat(frames, ignore_index=True)
             .sort_values(["ticker","date"])
             .reset_index(drop=True))

# minimal validations
assert rev_all["date"].notna().all(), "Bad dates."
assert rev_all["revenue"].notna().any(), "All revenue missing."
assert rev_all.duplicated(["ticker","date"]).sum() == 0, "Duplicate (ticker,date)."

raw_csv_path = RAW_DIR / f"api_revenue_banks_{STAMP}.csv"
rev_all.to_csv(raw_csv_path, index=False)
print("Saved RAW ->", raw_csv_path)


OK JPM: 7 rows
OK BAC: 6 rows
OK C: 6 rows
OK GS: 7 rows
OK MS: 6 rows
OK DB: 6 rows
Saved RAW -> C:\Users\User\bootcamp_Khushi_Khanna\project\data\raw\api_revenue_banks_20250825-1515.csv


In [12]:
def write_df(df: pd.DataFrame, path: Path):
    if path.suffix.lower() == ".csv":
        df.to_csv(path, index=False)
    elif path.suffix.lower() == ".parquet":
        df.to_parquet(path, index=False)  # needs pyarrow (installed)
    else:
        raise ValueError(f"Unsupported extension: {path.suffix}")

def read_df(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path, parse_dates=["date"])
    elif path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    else:
        raise ValueError(f"Unsupported extension: {path.suffix}")

proc_csv = PROC_DIR / f"banks_quarterly_revenue_{STAMP}.csv"
proc_par = PROC_DIR / f"banks_quarterly_revenue_{STAMP}.parquet"
write_df(rev_all, proc_csv)
write_df(rev_all, proc_par)

# reload checks
df_csv = read_df(proc_csv)
df_par = read_df(proc_par)
assert df_csv.shape == df_par.shape, "CSV vs Parquet shapes differ."
assert pd.api.types.is_datetime64_any_dtype(df_csv["date"]), "CSV reload lost 'date' dtype."
print("Processed :", proc_csv.name, "and", proc_par.name)


Processed : banks_quarterly_revenue_20250825-1502.csv and banks_quarterly_revenue_20250825-1502.parquet


In [22]:
from io import StringIO

WIKI_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
UA = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

html = requests.get(WIKI_URL, headers=UA, timeout=30).text
tables = pd.read_html(StringIO(html), flavor="lxml")  # <-- no FutureWarning

# pick the table with Symbol/Security headers
chosen = None
for t in tables:
    cols = [str(c) for c in t.columns]
    if any("Symbol" in c for c in cols) and any("Security" in c for c in cols):
        chosen = t
        break
if chosen is None:
    raise AssertionError("No suitable table found on the S&P 500 page.")

scrape_df = chosen.copy()
# clean headers
if isinstance(scrape_df.columns, pd.MultiIndex):
    scrape_df.columns = [" ".join([str(x) for x in tup if str(x) != "nan"]).strip()
                         for tup in scrape_df.columns]
scrape_df.columns = (pd.Series(scrape_df.columns)
                     .astype(str)
                     .str.replace(r"\[\d+\]", "", regex=True)
                     .str.replace("\xa0", " ")
                     .str.strip())
scrape_df = scrape_df.loc[:, ~scrape_df.columns.str.contains("^Unnamed", case=False, regex=True)]
scrape_df = scrape_df.dropna(how="all")

# quick validation + save (stays in project/data/raw/)
print("Scrape shape:", scrape_df.shape)
scrape_path = RAW_DIR / f"scrape_wiki_sp500_{STAMP}.csv"
scrape_df.to_csv(scrape_path, index=False)
print("Scraped RAW ->", scrape_path)


Scrape shape: (503, 8)
Scraped RAW -> C:\Users\User\bootcamp_Khushi_Khanna\project\data\raw\scrape_wiki_sp500_20250825-1515.csv


In [23]:
# strict paths under project/
from pathlib import Path
import sys, subprocess, importlib, datetime as dt

def ensure(pkg):
    try: importlib.import_module(pkg)
    except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ("pandas", "numpy", "yfinance", "pyarrow", "scikit-learn"):
    ensure(pkg)

import pandas as pd, numpy as np, yfinance as yf
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

# find actual project root (contains data/, src/, notebooks/)
def find_project_root():
    here = Path.cwd().resolve()
    for p in [here] + list(here.parents):
        if (p / "data").exists() and (p / "src").exists() and (p / "notebooks").exists():
            return p
    return Path.cwd().resolve().parent  # fallback: parent of notebooks

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR  = DATA_DIR / "raw"
PROC_DIR = DATA_DIR / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)

STAMP = dt.datetime.now().strftime("%Y%m%d-%H%M")

# universe
BANK_TICKERS = {
    "JPMorgan": "JPM",
    "Bank of America": "BAC",
    "Citigroup": "C",
    "Goldman Sachs": "GS",
    "Morgan Stanley": "MS",
    "Deutsche Bank": "DB",
}

print("ROOT:", PROJECT_ROOT)
print("RAW :", RAW_DIR)
print("PROC:", PROC_DIR)


ROOT: C:\Users\User\bootcamp_Khushi_Khanna\project
RAW : C:\Users\User\bootcamp_Khushi_Khanna\project\data\raw
PROC: C:\Users\User\bootcamp_Khushi_Khanna\project\data\processed


In [24]:
# pick latest saved revenue raw
rev_files = sorted(RAW_DIR.glob("api_revenue_banks_*.csv"))
assert rev_files, "No revenue CSV in project/data/raw. Run the ingestion cell first."
rev_path = rev_files[-1]

rev = pd.read_csv(rev_path, parse_dates=["date"])
# keep minimal cols; compute QoQ growth (%)
rev = (rev
       .sort_values(["ticker","date"])
       .assign(revenue=lambda d: pd.to_numeric(d["revenue"], errors="coerce"))
      )
rev["rev_qoq"] = (rev.groupby("ticker")["revenue"]
                    .pct_change()
                    .replace([np.inf, -np.inf], np.nan))

print(rev_path.name, "rows:", len(rev))
rev.tail(8)


api_revenue_banks_20250825-1515.csv rows: 38


C:\Users\User\AppData\Local\Temp\ipykernel_9244\3361767824.py:13: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change()


,date,revenue,source,bank,ticker,rev_qoq
30,2025-03-31,4.532700e+10,yfinance,JPMorgan,JPM,0.059265
31,2025-06-30,4.488200e+10,yfinance,JPMorgan,JPM,-0.009818
32,2024-03-31,NaN,yfinance,Morgan Stanley,MS,NaN
33,2024-06-30,1.402400e+10,yfinance,Morgan Stanley,MS,NaN
34,2024-09-30,1.433900e+10,yfinance,Morgan Stanley,MS,0.022461
35,2024-12-31,1.504300e+10,yfinance,Morgan Stanley,MS,0.049097
36,2025-03-31,1.651700e+10,yfinance,Morgan Stanley,MS,0.097986
37,2025-06-30,1.560400e+10,yfinance,Morgan Stanley,MS,-0.055276
